### Importing Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Importing Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground 3.1/train.csv').drop('id', axis=1)
df

# Data Visualizations

In [ ]:
plt.scatter(df['Latitude'], df['Longitude'], c=df['MedHouseVal'], s=0.1)

In [ ]:
df.corr()['MedHouseVal'].abs().sort_values(ascending=False)

### Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

from sklearn.preprocessing import MinMaxScaler
norm = MinMaxScaler().fit(X)
X_norm = pd.DataFrame(norm.transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2)

X_norm

### Model Selection

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import BayesianRidge
from xgboost.sklearn import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR

classifiers = [
    RandomForestRegressor(n_estimators=10),
    LinearRegression(),
    BayesianRidge(),
    XGBRegressor(objective ='reg:squarederror'),
    LGBMRegressor(),
    #SVR()
]

scorings = ['neg_mean_squared_error', 'r2']
results = pd.DataFrame(columns=scorings)

for clf in classifiers:
  clf.fit(X_train, y_train)
  scores = []
  for scoring in scorings:
    score = np.mean(cross_val_score(clf, X_test, y_test, cv=5, scoring=scoring))
    scores.append(np.round(score,4))
  name = clf.__class__.__name__
  results.loc[name] = scores

results

In [ ]:
plt.figure(figsize = (20, 6))
sns.barplot(x=results.index, y=-results['neg_mean_squared_error'], palette='viridis')
plt.xlabel('Model')
plt.ylabel('RMSE')
plt.title('RMSE of different models');

### Model Tuning

In [ ]:

import xgboost as xgb
from sklearn.metrics import log_loss
from sklearn.metrics import mean_squared_error as MSE
from sklearn.model_selection import GridSearchCV, StratifiedKFold

xgb_model = xgb.XGBRegressor(verbosity=0)

parameters = {'nthread':[1,3,10],
              'objective':['reg:squarederror'],
              'learning_rate': [0.1, 0.01],
              'max_depth': [1,5,10],
              #'min_child_weight': [1,2,3,10,12],
              #'silent': [0],
              #'subsample': [0.1,0.2,0.5,0.8],
              'colsample_bytree': [0.01, 0.1, 0.2, 0.5, 1],}
              #'n_estimators': [1000],
              #'missing':[-999]} # For Tuning

xgb_model = GridSearchCV(estimator=xgb_model, param_grid=parameters, n_jobs=1, refit=True, verbose=3)

xgb_model.fit(X_train, y_train)

print(xgb_model.best_score_)
#xgb_model.best_params_

In [ ]:
xgb_model.best_params_

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

params = {'colsample_bytree': 0.5,
          'learning_rate': 0.1,
          'max_depth': 10,
          'nthread': 1,
          'objective': 'reg:squarederror'}
#XGB = XGBRegressor(**xgb_model.best_params_)
XGB = XGBRegressor(**params)

XGB.fit(X_norm, y)

y_pred = XGB.predict(X_test)

val_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(val_rmse)

### Final Submissions

In [ ]:
submission = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground 3.1/test.csv')[['id']]
X_test = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground 3.1/test.csv').drop('id', axis=1)

X_final = pd.concat([X, X_test], ignore_index=True)
norm = MinMaxScaler().fit(X_final)
X_norm = pd.DataFrame(norm.transform(X_final), columns=X.columns)

X_train = X_norm[:X.shape[0]]
X_test = X_norm[X.shape[0]:]

XGB.fit(X_train, y)

submission['MedHouseVal'] = XGB.predict(X_test)
submission

In [ ]:
submission.to_csv('/content/drive/MyDrive/Kaggle Competitions/Playground 3.1/submission.csv', index=False)